## Interactive Components
- Adjust the layput of a dash application through components (Html and Core)
- Interact and connect to those components through the use of callbacks




##### Steps to create a callback for interactions
- Create a function to return some desired output
- Decorate that function with an @app.callback  decorator
- Set an Output to a component id
- Set an Input to a component id
- Connect the desired properties

In [103]:
import dash
import dash_core_components as dc
import dash_html_components as html
from dash.dependencies import Input, Output , State
import plotly.graph_objs as go
import numpy as np
import pandas as pd
import base64 

In [6]:
app = dash.Dash()

## Connecting Components with callbacks

In [13]:
app.layout = html.Div([
    dc.Input(id='my-id',value='Input Text',type='text'),
    html.Div(id='my-div',style={'border':'1px blue solid'}),   
])

In [16]:
@app.callback(Output(component_id='my-div',component_property='children'),[Input(component_id='my-id',component_property='value')])
def update_output_div(input_value):
    return "you entered: {}".format(input_value)

In [17]:
app.run_server()

## Dash callbacks for Graphs

In [21]:
df = pd.read_csv('Data/gapminderDataFiveYear.csv')
df.head()

,country,year,pop,continent,lifeExp,gdpPercap
0,Afghanistan,1952,8425333.0,Asia,28.801,779.445314
1,Afghanistan,1957,9240934.0,Asia,30.332,820.853030
2,Afghanistan,1962,10267083.0,Asia,31.997,853.100710
3,Afghanistan,1967,11537966.0,Asia,34.020,836.197138
4,Afghanistan,1972,13079460.0,Asia,36.088,739.981106


In [24]:
app = dash.Dash()
year_options= []
for year in df['year'].unique():
    year_options.append({'label':str(year),'value':year})
app.layout = html.Div([dc.Graph(id='graph'),dc.Dropdown(id='year-picker',options=year_options,value=df['year'].min())]),

In [29]:
@app.callback(Output('graph','figure'),[Input('year-picker','value')])
def update_figure(selected_year):
    filtered_df = df[df['year'] == selected_year]

    traces = []

    for continent_name in filtered_df['continent'].unique():
        df_by_continent = filtered_df[filtered_df['continent']==continent_name]
        traces.append(go.Scatter(
            x = df_by_continent['gdpPercap'],
            y = df_by_continent['lifeExp'],
            mode='markers',
            opacity=0.7,
            marker={'size':15},
            name=continent_name
        ))
    return {'data':traces,'layout':go.Layout(title='My Plot',xaxis={'title':'GDP Per Cap','type':'log'},yaxis={'title':'Life Expectency'})}
    

In [30]:
app.run_server()

## Multiple Inputs

In [31]:
app = dash.Dash()

In [32]:
df = pd.read_csv('data/mpg.csv')

In [33]:
features = df.columns
features

Index(['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
       'acceleration', 'model_year', 'origin', 'name'],
      dtype='object')

In [34]:
app.layout = html.Div([
    html.Div([
        dc.Dropdown(id='xaxis',
                    options=[{'label':i,'value':i} for i in features],
                    value='displacement')],
             style={'width':'48%','display':'inline-block'}
            ),
    html.Div([
        dc.Dropdown(
            id='yaxis',
            options=[{'label':i,'value':i} for i in features],
            value='mpg'
        )],
        style={'width':'48%','display':'inline-block'}
    ),
    dc.Graph(id='feature-graphic')
],style={'padding':10})

In [41]:
@app.callback(Output('feature-graphic','figure'),[Input('xaxis','value'),Input('yaxis','value')])
def update_graph(xaxis_name,yaxis_name):
    return {'data':[go.Scatter(x=df[xaxis_name],
                               y=df[yaxis_name],
                               text=df['name'],
                               mode='markers',
                               marker={'size':15,'opacity':0.5,'line':{'width':0.5,'color':'white'}}
                              )],
                    'layout':go.Layout(title='My Dashboard for MPG',
                    xaxis={'title':xaxis_name},
                    yaxis={'title':yaxis_name},
                    hovermode='closest'
                )
           }

In [40]:
app.run_server()

## Multiple Outputs

In [80]:
df = pd.read_csv('Data/wheels.csv')
df.head()

,wheels,color,image
0,1,red,redunicycle.jpg
1,1,blue,blueunicycle.jpg
2,1,yellow,yellowunicycle.jpg
3,2,red,redbicycle.jpg
4,2,blue,bluemotorcycle.jpg


In [81]:
app = dash.Dash()

In [82]:
def encode_image(image_file):
    encoded = base64.b64encode(open(image_file,'rb').read())
    return 'data:image/png;base64,{}'.format(encoded.decode())

In [83]:
app.layout = html.Div([
            dc.RadioItems(id='wheels',options=[{'label':i,'value':i} for i in df['wheels'].unique()],value=1),
            html.Div(id='wheels-output'),
            html.Hr(),
            dc.RadioItems(id='colors',options=[{'label':i,'value':i} for i in df['color'].unique()],value='yellow'),
            html.Div(id='colors-output'),
            html.Img(id='display-image',src='children',height=300)
],style={'fontFamily':'helvetica','fontSize':18})

In [84]:
@app.callback(Output('wheels-output','children'),[Input('wheels','value')])
def callback_a(wheels_value):
    return "You choose {}".format(wheels_value)

In [85]:
@app.callback(Output('colors-output','children'),[Input('colors','value')])
def callback_b(colors_value):
    return "You choose {}".format(colors_value)

In [86]:
@app.callback(Output('display-image','src'),[Input('wheels','value'),Input('colors','value')])
def callback_image(wheel,color):
    path = 'Data/images/'
    return encode_image(path+df[(df['wheels']==wheel) & (df['color']==color)]['image'].values[0])

In [87]:
app.run_server()

## Dashboard 

In [99]:
app = dash.Dash()

In [100]:
app.layout = html.Div([
    dc.RangeSlider(id='range-slider',min=-10,max=11,marks={i:str(i) for i in range(-10,11)},value=[-1,1]),
    html.H1(id='product')
])

In [101]:
@app.callback(Output('product','children'),[Input('range-slider','value')])
def update_value(value_list):
    return value_list[0] * value_list[1]

In [102]:
app.run_server()

## Controlling Callback with Dash State
- Dash offers the ability to store saved changes and send them back on command
- dash.dependecnies.State can be used
- It is like hitting a submit button on a form
- State() is added to the @app.callback along with an Input() and Output()
- The state is connected to a component id and a property to report back.
  

In [111]:
app = dash.Dash()

In [112]:
app.layout = html.Div([dc.Input(id='number-in',value=1,style={'fontSize':24}),
                       html.Button(id='submit-button',n_clicks=0,children='Submit Here',style={'fontsize':24}),
                       html.H1(id='number-out')])

In [113]:
@app.callback(Output('number-out','children'),[Input('submit-button','n_clicks')],[State('number-in','value')])
def output(n_clicks,number):
    return number

In [114]:
app.run_server()